# Lecture 8 Q4: L1-Regularized MSE Loss Contours

Explore how adding an L1 penalty deforms the MSE loss contours.

**Question:** Given the MSE contour and L1-norm contour for a 2D linear model, which shape best represents the L1-regularized loss?

Use the λ slider to blend the two contours and see the resulting shape.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

# MSE loss: elliptical contours centered near (1.5, 0.5)
theta_min = np.array([1.5, 0.5])
A = np.array([[3.0, 1.0], [1.0, 1.2]])  # Hessian (positive definite)

th1 = np.linspace(-1.0, 3.5, 400)
th2 = np.linspace(-1.5, 2.5, 400)
T1, T2 = np.meshgrid(th1, th2)

def mse_loss(t1, t2):
    d = np.stack([t1 - theta_min[0], t2 - theta_min[1]], axis=-1)
    return (d @ A * d).sum(axis=-1)

def l1_norm(t1, t2):
    return np.abs(t1) + np.abs(t2)

Z_mse = mse_loss(T1, T2)
Z_l1  = l1_norm(T1, T2)

# Normalise to [0,1] for blending
Z_mse_n = Z_mse / Z_mse.max()
Z_l1_n  = Z_l1  / Z_l1.max()

lam_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=2.0, step=0.05,
    description='λ', continuous_update=False,
    style={'description_width': 'initial'})

out = widgets.Output()

def render(*_):
    lam = lam_slider.value
    Z_reg = Z_mse_n + lam * Z_l1_n

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    levels = np.linspace(Z_mse_n.min(), np.percentile(Z_mse_n, 75), 10)

    # Panel 1: MSE contour
    axes[0].contourf(T1, T2, Z_mse_n, levels=20, cmap='Blues')
    axes[0].contour(T1, T2, Z_mse_n, levels=10, colors='white', linewidths=0.6, alpha=0.6)
    axes[0].set_title('MSE Loss\n(elliptical contours)', fontsize=11)
    axes[0].set_xlabel('θ₁'); axes[0].set_ylabel('θ₂')
    axes[0].axhline(0, color='k', lw=0.5); axes[0].axvline(0, color='k', lw=0.5)

    # Panel 2: L1 norm contour
    axes[1].contourf(T1, T2, Z_l1_n, levels=20, cmap='Oranges')
    axes[1].contour(T1, T2, Z_l1_n, levels=10, colors='white', linewidths=0.6, alpha=0.6)
    axes[1].set_title('L1 Norm  ‖θ‖₁\n(diamond contours)', fontsize=11)
    axes[1].set_xlabel('θ₁')
    axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)

    # Panel 3: Regularized loss
    axes[2].contourf(T1, T2, Z_reg, levels=20, cmap='Greens')
    axes[2].contour(T1, T2, Z_reg, levels=10, colors='white', linewidths=0.6, alpha=0.6)
    tag = f'λ={lam:.2f}'
    shape_note = 'elliptical' if lam < 0.3 else ('kink-distorted ellipse' if lam < 1.2 else 'strongly L1-distorted')
    axes[2].set_title(f'MSE + λ·‖θ‖₁  ({tag})\n→ {shape_note}', fontsize=11)
    axes[2].set_xlabel('θ₁')
    axes[2].axhline(0, color='k', lw=0.5); axes[2].axvline(0, color='k', lw=0.5)

    fig.suptitle(
        'L1 regularization adds axis-aligned kinks (corners) to the elliptical MSE contours — '
        'the correct answer is option A.',
        fontsize=10, y=0.02)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()

lam_slider.observe(render, 'value')
display(lam_slider, out)
render()